> 模型/algos/metrics/数理工具，哆啦A梦的工具箱（四次元空间袋）。学习科研方法：实验设计与 analysis（metircs definition） & writing

- Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?
    - RL（**RL-zero**） 仅仅是提升了采样效率（但重要的也是这一点，从实用的角度），但远未“榨干”基座模型的潜力。
    - RL 压缩了探索空间，导致推理边界（reasoning boundary）收窄。RLVR (math/coding) 可能只是在做 "Distribution Sharpening" 而非真正的 "Capability Uplift"。
    - RL 并没有让模型学会"做它以前不会做的题"，而是让模型"更倾向于输出它以前偶尔能做对的答案"。RL 实际上牺牲了多样性，导致对于那些低概率但能做对的题目（Exploration），RL 模型反而因为过拟合高奖励路径而彻底丧失了解题能力。
    - "RL shifted the probability distribution and increased the probability of an otherwise less common correct answer trajectory."
- analysis
    - why：token space (exploration space, base model 已经训得`非常非常好`了，ppl 很低，分布已经比较得尖峰，较难采样到新的 solution)
    - KL & temperature
- misc
    - https://github.com/LeapLabTHU/limit-of-RLVR/tree/main
    - RL learning dynamics
    - discussion

### metrics

$$
\text{pass}@k := \mathbb{E} \left[ 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}} \right]
$$
该公式计算的是从 n 个样本中随机抽取 k 个，"全都不正确"的概率的补集（“$k$ 次抽样全部错误的概率”，），即"至少有一个正确"的概率。作者通常设置 n=128,256,or 1024。*（这样做可以在 $n$ 固定（例如 $n=1024$）的情况下，低方差地估算出所有 $k < n$ 的 pass@k 曲线 8。）
$$
PPL_{Base}(Y_{RL}|x) = \exp \left( -\frac{1}{T} \sum_{t=1}^{T} \log P_{Base}(y_t|x, y_{<t}) \right)
$$
为了验证 RL 模型的输出是否"逃逸"出了 Base Model 的分布，作者计算 Base Model 对 RL 模型生成的回复 $Y_\text{RL}$ 的困惑度。如果 $PPL_{Base}(Y_{RL}|x)$ 很低，说明 RL 生成的内容在 Base Model 看来是"意料之中"的。
$$
\Delta_{SE} = \text{pass}@k_{\text{Base}} - \text{pass}@1_{\text{RL}}
$$

采样效率差距（Sampling Efficiency Gap, $\Delta_{SE}$） 是作者为了量化 RL 算法在"挖掘基座模型潜力"方面的有效性而提出的一个指标。$\text{pass@}k_{\text{Base}}$：基座模型采样 $k$ 次（论文中通常取 $k=256$）能做对的概率。作者将此作为该模型推理能力的**"潜能上限" (Upper Bound)**。现实情况 ($\Delta_{SE}$ 很大)：论文实验发现，目前的 RL 算法（如 PPO, GRPO 等）产生的 $\Delta_{SE}$ 依然很大（通常 > 40%）。例如：Base Model 采样 256 次能解决 70% 的题，但 RL Model 采样 1 次只能解决 30% 的题。中间这 40% 的差距，就是 $\Delta_{SE}$。（本文还对比了 PPO, Reinforce++, RLOO, ReMax, DAPO 等算法，发现它们在采样效率差距 ($\Delta_{SE}$) 上表现相似，都未能逼近 Base Model 的理论上限。）

### 计算细节

对于第 $i$ 个问题，已知总共有 $N$ 个样本，其中 $c_i$ 个是正确的。
代码中的 `pass_at_k(n, c, k)` 函数计算的是在该分布下，随机抽取 $k$ 个样本，至少有一个正确的概率 $P_i(k)$。
代码使用了以下公式（基于超几何分布的补集）：
$$
P_i(k) = \begin{cases}
1.0 & \text{if } N - c_i < k \\
1 - \prod_{j=N-c_i+1}^{N} \left( 1 - \frac{k}{j} \right) & \text{otherwise}
\end{cases}
$$

- 如果错误的样本总数 ($N - c_i$) 小于抽取的数量 $k$，那么无论怎么抽，必然会包含至少一个正确的样本，因此概率为 1。
- 代码中 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)) 计算的是“抽取的 $k$ 个样本全都是错误”的概率 $P(\text{fail})$，然后用 $1$ 减去它。
    - 实验中 $N$ 可以达到 1024； 

$$
pass@k = 1 - \frac{\binom{N-c_i}{k}}{\binom{N}{k}}
$$

In [1]:
import numpy as np
import math

def pass_at_k(n, c, k):
    if n - c < k: return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))
def pass_at_k2(n, c, k):
    if n - c < k: return 1.0
    return 1 - math.comb(n-c, k) / math.comb(n, k)

In [2]:
pass_at_k(5, 1, 3), pass_at_k2(5, 1, 3)

(np.float64(0.6), 0.6)

最终的 pass@k 指标是对所有 $Q$ 个问题的 $pass@k$ 概率求平均值。对于给定的 $k$，数据集上的平均 Pass@k 为：
$$
\text{Pass}@k = \frac{1}{Q} \sum_{i=1}^{Q} P_i(k)
$$

### Distillation

作者分析了 RL 模型生成的正确答案的 PPL。发现 $PPL_{base}(Y_{RL})$ 非常低，且随着训练进行逐渐降低。这说明 RL 模型生成的推理路径，完全包含在 Base Model 的分布内。RL 只是对 Base Model 的分布进行了"削尖" (Sharpening)，而非"拓展" (Expanding)。

Distillation 的不同： 与 RL 不同，作者发现 Distillation (蒸馏) 模型（如 DeepSeek-R1-Distill）的曲线是完全包围 Base Model 的。这说明蒸馏确实通过引入 Teacher 的知识，扩展了模型的推理边界。

### inference & eval

- math: `math/`
    - https://huggingface.co/datasets/math-ai/minervamath
        - 272 (272 * 32 -> 8704)
    - https://github.com/LeapLabTHU/limit-of-RLVR/blob/main/math/examples/math_eval/math_eval.py
- coding: `code/DeepCoder`
    - https://github.com/LeapLabTHU/limit-of-RLVR/blob/main/code/DeepCoder/verl/verl/trainer/main_generation_vllm2.py

In [3]:
# base model

In [4]:
from vllm import LLM, SamplingParams

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-25 10:39:53 [__init__.py:216] Automatically detected platform cuda.


In [5]:
import json
with open('./data/test.jsonl', 'r', encoding='utf-8') as f:
    examples = [json.loads(line) for line in f]
examples = sorted(examples, key=lambda x: x["idx"])

In [6]:
from tqdm import tqdm
from utils import extract_answer

In [1]:
# custom chat template, not tokenizer.apply_chat_template
# 是跟实际 rlvr 训练保持一致，不走默认的 chat template
prompt_temp = (
        "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
        "<|im_start|>user\n{input}\nPlease reason step by step, and put your final answer within \\boxed{{}}.<|im_end|>\n"
        "<|im_start|>assistant\n",
        "{output}",
        "\n\n",
    )

In [8]:
input_template, output_template, splitter = prompt_temp

In [9]:
samples = []
for example in tqdm(examples):
    idx = example['idx']
    gt_cot = example['solution']
    gt_ans = extract_answer(example['solution'], 'minerva_math')
    # print(f"raw: {example['solution']}")
    # print(f'processed: {gt_ans}')
    example['gt_ans'] = gt_ans
    full_prompt = input_template.format(input=example['problem']).strip(" ")
    sample = {'idx': idx, 'question': example['problem'], 'gt_cot': gt_cot, 'gt': 'gt_ans', 'prompt': full_prompt}
    sample['type'] = example['type']
    samples.append(sample)

100%|██████████████████████████████████████████████████████████████████████████████████████| 272/272 [00:00<00:00, 74172.73it/s]


In [10]:
n_sampling = 32
input_prompts = [sample['prompt'] for sample in samples for _ in range(n_sampling)]

In [11]:
assert len(input_prompts) == 32 * 272

In [12]:
stop_words = ["</s>", "<|im_end|>", "<|endoftext|>"]
stop_words.extend(["assistant", "user", "_end", "_start"])
stop_token_ids = [151645, 151643]

In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
llm = LLM(model='/data/zch/weights/Qwen2.5-7B',
          trust_remote_code=True, 
          seed=1, 
          gpu_memory_utilization=0.8,
          max_model_len=4096)

INFO 11-25 10:39:56 [utils.py:233] non-default args: {'trust_remote_code': True, 'seed': 1, 'max_model_len': 4096, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': '/data/zch/weights/Qwen2.5-7B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 11-25 10:39:56 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-25 10:39:56 [model.py:1510] Using max model len 4096


...

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:03 [parallel_state.py:1208] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:03 [topk_topp_sampler.py:55] Using FlashInfer for top-p & top-k sampling.
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:03 [gpu_model_runner.py:2602] Starting to load model /data/zch/weights/Qwen2.5-7B...
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:03

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  2.39it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:00,  2.20it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  2.09it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.01it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.07it/s]
(EngineCore_DP0 pid=1314415) 


(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:06 [default_loader.py:267] Loading weights took 2.07 seconds
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:06 [gpu_model_runner.py:2653] Model loading took 14.2717 GiB and 2.258156 seconds
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:09 [backends.py:548] Using cache directory: /home/zhangchunhui/.cache/vllm/torch_compile_cache/dae78d5a71/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:09 [backends.py:559] Dynamo bytecode transform time: 3.01 s
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:11 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.180 s
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:11 [monitor.py:34] torch.compile takes 3.01 s in total
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:51 [gpu_worker.py:298] Available KV cache memory: 3.52 GiB
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:51 [kv_cache_utils.py:1087] GPU KV cache size: 65,888 tokens
(

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 22.57it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 30.67it/s]


(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:56 [gpu_model_runner.py:3480] Graph capturing finished in 5 secs, took 2.52 GiB
(EngineCore_DP0 pid=1314415) INFO 11-25 10:40:56 [core.py:210] init engine (profile, create kv cache, warmup model) took 49.70 seconds
INFO 11-25 10:40:56 [llm.py:306] Supported_tasks: ['generate']


In [14]:
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=16000,
    n=1,
    # stop=stop_words,
    # stop_token_ids=stop_token_ids,
)

In [15]:
print(input_prompts[0])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Each of the two Magellan telescopes has a diameter of $6.5 \mathrm{~m}$. In one configuration the effective focal length is $72 \mathrm{~m}$. Find the diameter of the image of a planet (in $\mathrm{cm}$ ) at this focus if the angular diameter of the planet at the time of the observation is $45^{\prime \prime}$.
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>assistant



In [16]:
output = llm.generate(input_prompts[:1], sampling_params)

Processed prompts: 100%|████████████████████| 1/1 [00:09<00:00,  9.71s/it, est. speed input: 12.26 toks/s, output: 61.20 toks/s]


In [17]:
output[0].outputs[0].text, output[0].outputs[0].finish_reason

("To solve this problem, we need to use the relationship between the angular diameter of an object, its actual diameter, and its distance from the observer. Here's the step-by-step solution:\n\n1. **Convert the angular diameter from arcseconds to radians**:\n   The angular diameter is given as 45 arcseconds. Since there are 3600 arcseconds in a degree and π radians in 180 degrees, we can convert arcseconds to radians using the formula:\n   \\[\n   \\text{Angular diameter in radians} = \\frac{\\text{Angular diameter in arcseconds}}{3600} \\times \\frac{\\pi}{180}\n   \\]\n   Substituting the given value:\n   \\[\n   \\text{Angular diameter in radians} = \\frac{45}{3600} \\times \\frac{\\pi}{180} = \\frac{\\pi}{240}\n   \\]\n\n2. **Use the small angle approximation**:\n   For small angles, the tangent of the angle is approximately equal to the angle itself in radians. Therefore, the ratio of the actual diameter of the planet (D) to its distance from the observer (L) is approximately equa

In [18]:
outputs = llm.generate(input_prompts[:100], sampling_params)

Processed prompts: 100%|█████████████| 100/100 [00:22<00:00,  4.51it/s, est. speed input: 675.20 toks/s, output: 2681.48 toks/s]


In [19]:
outputs = sorted(
    outputs, key=lambda x: int(x.request_id)
)  # sort outputs by request_id
outputs = [
    (output.outputs[0].text, output.outputs[0].finish_reason)
    for output in outputs
]

In [20]:
# [o[1] for o in outputs]

```
{
    "num_samples": 272,
    "num_scores": 8704,
    "timeout_samples": 19,
    "empty_samples": 23,
    "acc": 23.2,
    "pass_acc": 58.5,
    "pass@k": {
        "1": 23.2,
        "2": 32.3,
        "4": 40.9,
        "8": 48.2,
        "16": 53.9,
        "32": 58.5
    },
    "type_acc": {
        "Differential Equations (18.03 Spring 2010)": 47.9,
        "Dynamics and Control (2.003 Spring 2005)": 30.8,
        "Ecology I (1.018J Fall 2009)": 0.0,
        "Information and Entropy (6.050J Spring 2008)": 0.0,
        "Introduction to Astronomy (8.282J Spring 2006)": 15.1,
        "Introduction to Solid State Chemistry (3.091 Fall 2010)": 11.3,
        "Physical Chemistry (5.61 Fall 2017)": 9.1,
        "Principles of Microeconomics (14.01 Fall 2011)": 50.0,
        "Relativity (8.033 Fall 2006)": 18.2
    },
    "time_use_in_second": 1018.528128862381,
    "time_use_in_minite": "16:58"
}
```

#### PPL

$$
\sum_{t=1}^{T}\log P(y_t|\dots) = \log\left(\prod_{t=1}^{T} P(y_t|\dots)\right) = \log(P_{\text{joint}})
$$
$$
\frac{1}{T} \log(P_{\text{joint}}) = \log\left(P_{\text{joint}}^{\frac{1}{T}}\right)
$$
$$
-\log\left(P_{\text{joint}}^{\frac{1}{T}}\right) = \log\left(\frac{1}{P_{\text{joint}}^{\frac{1}{T}}}\right)
$$
$$
PPL = \exp\left(-\frac{1}{T}\sum_{t=1}^{T}\log P(y_t|\dots)\right)=\exp\left(\log\left(\frac{1}{P_{\text{joint}}^{\frac{1}{T}}}\right)\right) = \frac{1}{\sqrt[T]{P_{\text{joint}}}}
$$
- 联合概率几何平均的倒数；
    - 几何平均概率：表示模型平均预测每个 token 的概率是多少。例如，平均每个词猜对的概率是 0.5。
        - 避免联合概率值过小的问题，尤其是随着句子变长，也平滑长度带来的影响，句子变长天然地联合概率变低；
        - log p 的算术平均 = p 的几何平均
    - PPL（倒数）：表示模型在每一步面对的选择“分支系数”是多少。如果平均概率是 0.5，PPL 就是 2，意味着模型在每一步平均相当于在 2 个选项中犹豫。
        - 困惑度为 k 意味着模型在预测每个词时，其困惑程度相当于在 k 种可能性中均匀且独立地进行选择。
- 词表 $|V|$，瞎猜的 ppl baseline
    - $P(w_t) = \frac{1}{|V|}$
    - 由于每个位置的概率都是 $\frac{1}{|V|}$，那么它们的几何平均值依然是 $\frac{1}{|V|}$。
    - $\text{Geometric Mean} = \sqrt[T]{\prod_{t=1}^{T} \frac{1}{|V|}} = \frac{1}{|V|}$
    - $PPL = \frac{1}{\text{Geometric Mean}} = |V|$
- $L=\log(PPL), PPL=\exp(L)$
    - 越低越好

In [21]:
from transformers import AutoTokenizer

In [22]:
tokenizer = llm.get_tokenizer()

In [23]:
input_prompts[0] == tokenizer.decode(tokenizer.encode(input_prompts[0], add_special_tokens=False))

True

In [24]:
def calculate_perplexity(llm, prompts, responses):
    
    # 拼接 Prompt 和 Response
    # VLLM 会把整段文本作为 "prompt" 处理，并返回每个 token 的 logprobs
    full_texts = [p + r for p, r in zip(prompts, responses)]
    
    # max_tokens=1: 我们只做评估，不生成新内容的 token
    # prompt_logprobs=1: 请求返回输入文本中每个位置的 logprob, 每个位置输出多少 top-k 的 token & log p
    eval_params = SamplingParams(
        prompt_logprobs=1,
        max_tokens=1,
        temperature=1.0,
        # detrimental_token_id=-1
    )

    # foward pass
    print(f"Running PPL evaluation on {len(full_texts)} samples...")
    outputs = llm.generate(full_texts, eval_params)
    
    ppl_scores = []
    tokenizer = llm.get_tokenizer()
    
    for i, output in enumerate(outputs):
        prompt_text = prompts[i]
        
        # 获取完整的 logprobs 序列
        token_logprobs = output.prompt_logprobs
        
        # 确定 Response 的起始位置 (Masking)
        prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        # print(tokenizer.decode(prompt_token_ids))
        # break
        start_idx = len(prompt_token_ids)
        
        if start_idx >= len(token_logprobs):
            print(f"Warning: Sample {i} length mismatch. Prompt tokens: {start_idx}, Total logprobs: {len(token_logprobs)}")
            ppl_scores.append(float('nan'))
            continue

        # 提取 Response 部分的 Log Probability
        response_log_probs = []

        print(output.prompt_token_ids[start_idx:])        
        print(token_logprobs[start_idx:])
        break
        
        # 从 start_idx 开始遍历 (即只计算 Response 部分的概率)
        for step, logprob_dict in enumerate(token_logprobs[start_idx:], start=start_idx):
            if logprob_dict is None: 
                continue
            
            # 获取该位置实际的 token id
            actual_token_id = output.prompt_token_ids[step]
            
            assert actual_token_id in logprob_dict
            response_log_probs.append(logprob_dict[actual_token_id].logprob)

        # 计算 Perplexity
        if response_log_probs:
            avg_neg_log_likelihood = -np.mean(response_log_probs)
            ppl = np.exp(avg_neg_log_likelihood)
            ppl_scores.append(ppl)
        else:
            ppl_scores.append(float('nan'))

    return ppl_scores

In [25]:
ppl_results = calculate_perplexity(llm, input_prompts[:100], [o[0] for o in outputs])

Running PPL evaluation on 100 samples...


Processed prompts: 100%|██████████████| 100/100 [00:07<00:00, 12.79it/s, est. speed input: 9500.84 toks/s, output: 12.79 toks/s]

[1249, 11625, 419, 3491, 11, 582, 1184, 311, 990, 279, 7286, 315, 20314, 1379, 323, 279, 5888, 315, 264, 55825, 13, 576, 20314, 1379, 315, 279, 11580, 518, 279, 882, 315, 21930, 374, 2661, 304, 15580, 17403, 11, 323, 582, 1184, 311, 5508, 432, 311, 50784, 311, 990, 432, 304, 1039, 28117, 13, 576, 23033, 315, 279, 2168, 315, 279, 11580, 518, 279, 5244, 315, 279, 55825, 646, 387, 16588, 1667, 279, 14806, 1447, 78045, 1124, 1318, 90, 35, 36044, 315, 279, 2168, 92, 284, 1124, 37018, 35702, 1318, 90, 68359, 1379, 304, 50784, 92, 1124, 15136, 1124, 1318, 90, 37, 3683, 3084, 3417, 90, 17, 92, 1124, 2533, 5338, 11, 1077, 594, 5508, 279, 20314, 1379, 504, 15580, 17403, 311, 50784, 1447, 78045, 220, 19, 20, 1124, 1318, 90, 15580, 17403, 92, 284, 220, 19, 20, 1124, 15136, 1124, 37018, 35702, 2493, 15170, 16, 23, 15, 1124, 15136, 220, 18, 21, 15, 15, 92, 1124, 1318, 90, 50784, 92, 1124, 2533, 5847, 11, 582, 646, 11047, 279, 23033, 315, 279, 2168, 1667, 279, 2661, 41099, 3084, 315, 400, 22, 17, 112

In [26]:
ppl_results

[]

#### output entropy

$$
H(x) = -\sum p(x) \log p(x)
$$
- 所有的位置，Average Token Entropy

In [27]:
def calculate_metrics(llm, prompts, responses, top_k_for_entropy=20):
    """
    计算 Perplexity (PPL) 和 Output Entropy (输出熵)
    """
    
    # 拼接 Prompt 和 Response
    full_texts = [p + r for p, r in zip(prompts, responses)]
    
    # 关键修改 1: prompt_logprobs 必须设大一点 (例如 20，vllm 支持最高是20)，
    # 因为计算熵需要分布信息 (Sum(p * log p))，仅 Top-1 无法计算熵。
    eval_params = SamplingParams(
        prompt_logprobs=top_k_for_entropy, 
        max_tokens=1,
        temperature=1.0,
    )

    print(f"Running evaluation on {len(full_texts)} samples...")
    outputs = llm.generate(full_texts, eval_params)
    
    ppl_scores = []
    entropy_scores = [] # 新增：存储每个样本的平均熵
    
    tokenizer = llm.get_tokenizer()
    
    for i, output in enumerate(outputs):
        prompt_text = prompts[i]
        
        # 获取完整的 logprobs 序列
        # 这是一个 list[dict], 每个 dict 包含该位置 Top-K token 的 {id: Logprob对象}
        token_logprobs = output.prompt_logprobs
        
        # 确定 Response 的起始位置 (Masking Prompt)
        prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        start_idx = len(prompt_token_ids)
        
        if start_idx >= len(token_logprobs):
            print(f"Warning: Sample {i} length mismatch.")
            ppl_scores.append(float('nan'))
            entropy_scores.append(float('nan'))
            continue

        response_log_probs = [] # 用于计算 PPL (针对实际发生的 token)
        step_entropies = []     # 用于计算 Entropy (针对模型预测的分布)

        # 从 start_idx 开始遍历 (只计算 Response 部分)
        for step, logprob_dict in enumerate(token_logprobs[start_idx:], start=start_idx):
            if logprob_dict is None: 
                continue
            
            # 获取该位置实际存在的 token id (Ground Truth)
            actual_token_id = output.prompt_token_ids[step]
            
            # 注意：如果 prompt_logprobs 较小(如20)且模型预测非常偏，
            # 实际的 token 可能不在返回的 Top-K 字典里。
            assert actual_token_id in logprob_dict
            response_log_probs.append(logprob_dict[actual_token_id].logprob)

            # Shannon Entropy H(x) = - Sum(p * log(p))
            # 从 logprob_dict 中提取所有 Top-K 的 logprobs
            # logprob_dict.values() 是 Logprob 对象，包含 .logprob 属性
            current_step_logprobs = np.array([lp.logprob for lp in logprob_dict.values()])
            
            # 将 log_prob 转为 prob
            current_step_probs = np.exp(current_step_logprobs)
            
            # 计算当前步的熵: - sum(p * log_p)
            # 注意：因为只是 Top-K 近似，概率和不一定为 1，但通常用于衡量不确定性已足够
            current_entropy = -np.sum(current_step_probs * current_step_logprobs)
            step_entropies.append(current_entropy)

        # 汇总 PPL
        if response_log_probs:
            avg_neg_log_likelihood = -np.mean(response_log_probs)
            ppl = np.exp(avg_neg_log_likelihood)
            ppl_scores.append(ppl)
        else:
            ppl_scores.append(float('nan'))

        # 汇总 Entropy (取平均值，即 Average Token Entropy)
        if step_entropies:
            avg_entropy = np.mean(step_entropies)
            entropy_scores.append(avg_entropy)
        else:
            entropy_scores.append(float('nan'))

    return ppl_scores, entropy_scores

In [30]:
ppl, entropy = calculate_metrics(llm, input_prompts[:100], [o[0] for o in outputs])

Running evaluation on 100 samples...


Processed prompts: 100%|██████████████| 100/100 [00:08<00:00, 11.62it/s, est. speed input: 8628.84 toks/s, output: 11.62 toks/s]


In [32]:
np.mean(entropy)

np.float64(0.265038162663188)

[rank0]:[W1125 10:47:06.267532195 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


ERROR 11-25 10:47:06 [core_client.py:564] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
